# 🚗 01_target_and_quality_analysis: Data Quality & Target Distribution
This notebook follows the transition from **Raw Data** to **Cleaned Data**. We perform a dual analysis:
1. **Quality Audit:** Identifying "garbage" in the raw data to justify our validation rules.
2. **Target Analysis:** Examining the cleaned distribution of `trip_duration` to guide our model selection and loss function.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib
import sys

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load a significant sample of raw data to identify outliers
RAW_PATH = pathlib.Path("../data/raw/NYC.csv")
df_raw = pd.read_csv(RAW_PATH, nrows=200_000, parse_dates=["pickup_datetime"])
df_raw.head()

## 1. Target Variable Noise
We check the raw `trip_duration` to see if there are physically impossible values (e.g., negative duration or trips lasting multiple days).

In [ ]:
plt.figure(figsize=(12, 4))
sns.boxplot(x=df_raw['trip_duration'], color='coral')
plt.title("Raw Trip Duration - Identifying the Noise Floor and Ceiling")
plt.show()

print(f"Min duration: {df_raw['trip_duration'].min()}s")
print(f"Max duration: {df_raw['trip_duration'].max()}s")

## 2. Coordinate Sanity Check
We plot the raw coordinates to see if there are points far outside the NYC bounding box.

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(df_raw['pickup_longitude'], df_raw['pickup_latitude'], s=1, alpha=0.1)
plt.title("Raw Pickup Coordinates")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

### ✍️ Engineering Inferences:
1. **Duration Rules:** We see values $\le 0$ and values $> 14,400$ (4 hours). We must drop these.
2. **Coordinate Rules:** We see points far outside NYC. We must define a bounding box (approx 40.5-42.0 N, -75.0 to -72.0 W).
3. **Nulls:** Checking `df_raw.isnull().sum()` reveals which columns are unreliable.

## 3. Distribution of Cleaned Target Variable
Now we apply our validation rules to see the "Cleaned" distribution. This determines if the target is skewed and if we should use a log-transformation.

In [ ]:
# Import the validation logic to create the cleaned set
sys.path.append("..")
from src.data.validate import validate_data

df_clean, report = validate_data(df_raw)
df_clean['target'] = df_clean['trip_duration']

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df_clean['target'], kde=True, color='green')
plt.title('Cleaned Trip Duration Distribution')
plt.xlabel('Seconds')

plt.subplot(1, 2, 2)
sns.boxplot(x=df_clean['target'], color='lightgreen')
plt.title('Cleaned Trip Duration Boxplot')
plt.xlabel('Seconds')

plt.tight_layout()
plt.show()

print("\nCleaned Distribution Statistics:")
display(df_clean['target'].describe())

### ✍️ Final Target Inferences:
1. **Skewness:** Even after cleaning, the target remains right-skewed.
2. **Metric Choice:** This confirms that **MAE (Mean Absolute Error)** is a better primary metric than MSE, as it is less sensitive to the remaining long-tail trips.
3. **Strategy:** We will use models capable of handling skewed distributions (Tree-based ensembles) rather than assuming normality.